```
 Audio (WAV/MP3)
       ↓
  [Whisper] ← Transformer de OpenAI (Speech-to-Text)
       ↓
  Texto transcrito
       ↓
  [DistilBERT] ← Transformer (Clasificación de Sentimiento)
       ↓
   Positivo /  Negativo
```

In [30]:
# Instalación de librerias
!pip install transformers -q
!pip install openai-whisper -q
!pip install torch torchaudio -q
!pip install soundfile librosa -q
!pip install gtts -q  # Para generar audios de prueba desde texto

# ffmpeg es necesario para Whisper
!apt-get install -y ffmpeg -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [31]:
#Importamos librerias
import torch
import whisper
import numpy as np
from gtts import gTTS
from transformers import pipeline
from IPython.display import Audio, display
import os

In [32]:
#Utilizamos gTTS (Google Text to Speech) para generar audios de ejemplo
#Frases con sentimientos variados para probar el pipeline
frases_prueba = [
    {"texto": "Estoy completamente encantado de estar aqui y por la oportunidad", "sentimiento_esperado": "POSITIVE",
     "archivo": "audio_positivo.mp3"},

    {"texto": "Me voy, no valoran mi trabajo, me largo, fuera", "sentimiento_esperado": "NEGATIVE",
     "archivo": "audio_negativo.mp3"},

    {"texto": "Llegue a mis metas y resultados de este mes. NO soloeso, sino que los llegue a superar", "sentimiento_esperado": "POSITIVE",
     "archivo": "audio_feliz.mp3"},
]

# Generar archivos de audio
print('Generando audios de prueba\n')
for frase in frases_prueba:
    tts = gTTS(text=frase['texto'], lang='es', slow=False)
    tts.save(frase['archivo'])
    print(f" Creado: {frase['archivo']}")
    print(f"   Texto: '{frase['texto']}'")
    print(f"   Sentimiento esperado: {frase['sentimiento_esperado']}\n")

Generando audios de prueba

 Creado: audio_positivo.mp3
   Texto: 'Estoy completamente encantado de estar aqui y por la oportunidad'
   Sentimiento esperado: POSITIVE

 Creado: audio_negativo.mp3
   Texto: 'Me voy, no valoran mi trabajo, me largo, fuera'
   Sentimiento esperado: NEGATIVE

 Creado: audio_feliz.mp3
   Texto: 'Llegue a mis metas y resultados de este mes. NO soloeso, sino que los llegue a superar'
   Sentimiento esperado: POSITIVE



In [33]:
# Podemos escuchar los audios
print(' Audio 1 - Sentimiento POSITIVO:')
display(Audio('audio_positivo.mp3'))

print('\n Audio 2 - Sentimiento NEGATIVO:')
display(Audio('audio_negativo.mp3'))

print('\n Audio 3 - Sentimiento POSITIVO:')
display(Audio('audio_feliz.mp3'))

 Audio 1 - Sentimiento POSITIVO:



 Audio 2 - Sentimiento NEGATIVO:



 Audio 3 - Sentimiento POSITIVO:


## Arquitectua de transformeres

### Modelo 1: Whisper (Speech-to-Text)
- Arquitectura: **Encoder-Decoder Transformer**

### Modelo 2: DistilBERT (Clasificación de Sentimientos)
- Arquitectura: **Encoder-only Transformer**

In [34]:
# --- MODELO 1: Whisper para transcripción de audio ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(' Cargando Whisper')
modelo_whisper = whisper.load_model('base')  # Opciones varaidas: tiny, base, small, medium, large
print(' Whisper cargado correctamente\n')

# --- MODELO 2: DistilBERT para análisis de sentimiento ---
print(' Cargando DistilBERT para análisis de sentimientos')
clasificador_sentimiento = pipeline(
    task='sentiment-analysis',
    # Cambiamos a un modelo de sentimiento en español público
    model='finiteautomata/beto-sentiment-analysis',
    device=0 if device == 'cuda' else -1  # 0 = GPU, -1 = CPU
)

 Cargando Whisper
 Whisper cargado correctamente

 Cargando DistilBERT para análisis de sentimientos


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: finiteautomata/beto-sentiment-analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
#Funcion que encadena los dos tranformers
def analizar_sentimiento_desde_audio(ruta_audio, verbose=True):
    """
    Ruta : Audio → Transcripción (Whisper) → Sentimiento (DistilBERT)

    Args:
        ruta_audio (str): Ruta al archivo de audio (.mp3, .wav, etc.)
        verbose (bool): Si True, imprime el proceso paso a paso

    Returns:
        dict: {'transcripcion': str, 'sentimiento': str, 'confianza': float}
    """
    print(f' Procesando: {ruta_audio}')
    # PASO 1: Transcripción de audio a texto con Whisper
    # El Encoder de Whisper convierte el espectrograma de mel
    # en representaciones contextuales; el Decoder genera texto
    if verbose:
        print('\n[1/2]   Transcribiendo audio con Whisper')

    # Cambiamos el idioma de transcripción a español ('es')
    resultado_whisper = modelo_whisper.transcribe(ruta_audio, language='es')
    texto_transcrito = resultado_whisper['text'].strip()

    if verbose:
        print(f'     Texto transcrito: "{texto_transcrito}"')
    # PASO 2: Clasificación de sentimiento con DistilBERT
    # El token [CLS] al final del Encoder captura el sentido
    # global de la oración → se pasa a una capa de clasificación
    if verbose:
        print('\n[2/2]  Analizando sentimiento con DistilBERT')

    resultado_bert = clasificador_sentimiento(texto_transcrito)[0]
    etiqueta = resultado_bert['label']   # 'POSITIVE' o 'NEGATIVE'
    confianza = resultado_bert['score']  # valor entre 0 y 1

    # Ajustamos las etiquetas porque el modelo en español puede usar 'POS'/'NEG'
    # Declaramos que 'POS' y 'PN' son positivos, y 'NEG' y 'NEU' son negativos para simplificar
    if etiqueta == 'POS' or etiqueta == 'P':
        etiqueta = 'POSITIVE'
    elif etiqueta == 'NEG' or etiqueta == 'N':
        etiqueta = 'NEGATIVE'
    else: # Si hay etiquetas como NEU, asumimos que no es positivo
        etiqueta = 'NEGATIVE'

    emoji = ':)' if etiqueta == 'POSITIVE' else ':('
    barra = '█' * int(confianza * 20) + '░' * (20 - int(confianza * 20))

    if verbose:
        print(f'\n{'=' * 55}')
        print(f'  RESULTADO FINAL')
        print(f'{'=' * 55}')
        print(f'  {emoji}  Sentimiento : {etiqueta}')
        print(f'   Confianza  : {confianza:.2%}')
        print(f'  [{barra}]')
        print(f'{'=' * 55}\n')

    return {
        'archivo': ruta_audio,
        'transcripcion': texto_transcrito,
        'sentimiento': etiqueta,
        'confianza': confianza
    }

In [36]:
# Procesar todos los audios de prueba
resultados = []

for frase in frases_prueba:
    resultado = analizar_sentimiento_desde_audio(frase['archivo'])
    resultado['sentimiento_esperado'] = frase['sentimiento_esperado']
    resultados.append(resultado)

 Procesando: audio_positivo.mp3

[1/2]   Transcribiendo audio con Whisper


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


     Texto transcrito: "Estoy completamente encantado de estar aquí y por la oportunidad."

[2/2]  Analizando sentimiento con DistilBERT

  RESULTADO FINAL
  :)  Sentimiento : POSITIVE
   Confianza  : 99.87%
  [███████████████████░]

 Procesando: audio_negativo.mp3

[1/2]   Transcribiendo audio con Whisper


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


     Texto transcrito: "Me voy, no valora en mi trabajo, me largo, fuera."

[2/2]  Analizando sentimiento con DistilBERT

  RESULTADO FINAL
  :(  Sentimiento : NEGATIVE
   Confianza  : 99.89%
  [███████████████████░]

 Procesando: audio_feliz.mp3

[1/2]   Transcribiendo audio con Whisper


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


     Texto transcrito: "Llegue a mis metas y resultados de este mes. No solo eso, sino que los llegue a superar."

[2/2]  Analizando sentimiento con DistilBERT

  RESULTADO FINAL
  :)  Sentimiento : POSITIVE
   Confianza  : 97.77%
  [███████████████████░]



In [37]:
print(' RESUMEN DE RESULTADOS')
print(f'  {'Archivo':<22} {'Transcripción':<35} {'Pred':>8} {'Conf':>6}')

aciertos = 0
for r in resultados:
    correcto = r['sentimiento'] == r['sentimiento_esperado']
    if correcto:
        aciertos += 1
    icono = 'Si' if correcto else 'No'
    transcripcion_corta = r['transcripcion'][:32] + '...' if len(r['transcripcion']) > 32 else r['transcripcion']
    print(f"  {icono} {r['archivo']:<20} {transcripcion_corta:<35} {r['sentimiento']:>8} {r['confianza']:>5.1%}")

print('=' * 70)
print(f'  Precisión: {aciertos}/{len(resultados)} ({aciertos/len(resultados):.0%})')

 RESUMEN DE RESULTADOS
  Archivo                Transcripción                           Pred   Conf
  Si audio_positivo.mp3   Estoy completamente encantado de... POSITIVE 99.9%
  Si audio_negativo.mp3   Me voy, no valora en mi trabajo,... NEGATIVE 99.9%
  Si audio_feliz.mp3      Llegue a mis metas y resultados ... POSITIVE 97.8%
  Precisión: 3/3 (100%)
